# 类别不平衡：Focal Loss、阈值与业务成本

**面试问题：正例只有少数时，为什么 Accuracy 和固定 0.5 阈值不够，Focal Loss 怎么实现？**

## 回答主线

1. 类别不平衡首先影响指标和决策阈值，其次才是 Loss。
2. 只报 Accuracy 会奖励全部预测为负类，应优先看 Recall、Precision、PR-AUC 和业务成本。
3. 加权 BCE 提高少数类贡献，Focal Loss 再用 `(1-p_t)^gamma` 降低易样本权重。
4. Alpha 与 Gamma 必须在验证集调，不是越大越好。
5. 模型训练目标与上线阈值是两个决策：同一概率可以按漏报/误报成本选择不同阈值。
6. 概率校准、时间漂移和分群阈值仍需单独验证。

## 真实案例

35 笔支付交易中只有 5 笔欺诈，单一风险特征有重叠。我们用 PyTorch 从零训练一维 Logistic 模型，比较普通 BCE 和手写 Focal Loss，打印每个样本的 `p_t` 与权重，再按漏报成本 8、误报成本 1 搜索阈值。使用可读的离线教学数据解释机制，指标不能外推为线上收益。

### 输入预览：30 个正常交易与 5 个欺诈交易

In [1]:
import torch  # 导入 PyTorch 以手写可微 Focal Loss 和训练循环。

torch.manual_seed(19)  # 固定参数初始化保证结果可复现。
negative_risk = torch.tensor([0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.22, 0.24, 0.27, 0.30, 0.32, 0.34, 0.36, 0.38, 0.40, 0.42, 0.44, 0.46, 0.48, 0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.62, 0.64, 0.68, 0.72], dtype=torch.float32)  # 构造三十个正常风险分数。
positive_risk = torch.tensor([0.55, 0.61, 0.67, 0.78, 0.90], dtype=torch.float32)  # 构造五个与正常样本重叠的欺诈风险分数。
features = torch.cat([negative_risk, positive_risk]).unsqueeze(1)  # 合并为三十五行一列特征矩阵。
labels = torch.cat([torch.zeros(len(negative_risk)), torch.ones(len(positive_risk))])  # 构造 30:5 不平衡标签。
print(f"样本数={len(labels)}，正例={int(labels.sum())}，负例={int((1 - labels).sum())}，正例率={labels.mean().item():.1%}")  # 输出类别分布。
print("欺诈风险：", positive_risk.tolist())  # 展示少数类位置。
print("与欺诈重叠的正常风险：", negative_risk[negative_risk >= positive_risk.min()].tolist())  # 展示任务不可完全分离。

样本数=35，正例=5，负例=30，正例率=14.3%
欺诈风险： [0.550000011920929, 0.6100000143051147, 0.6700000166893005, 0.7799999713897705, 0.8999999761581421]
与欺诈重叠的正常风险： [0.5600000023841858, 0.5799999833106995, 0.6000000238418579, 0.6200000047683716, 0.6399999856948853, 0.6800000071525574, 0.7200000286102295]


## Baseline 基线：Accuracy 最优的全负类与普通 BCE

In [2]:
all_negative = torch.zeros_like(labels)  # 构造永远预测正常的朴素分类器。
all_negative_accuracy = (all_negative == labels).float().mean().item()  # 计算虚高 Accuracy。
all_negative_recall = 0.0  # 全负类无法召回任何欺诈。

def binary_metrics(probabilities, targets, threshold):  # 从概率和阈值计算混淆矩阵指标。
    predictions = (probabilities >= threshold).float()  # 应用业务阈值。
    true_positive = int(((predictions == 1) & (targets == 1)).sum())  # 统计 TP。
    false_positive = int(((predictions == 1) & (targets == 0)).sum())  # 统计 FP。
    false_negative = int(((predictions == 0) & (targets == 1)).sum())  # 统计 FN。
    true_negative = int(((predictions == 0) & (targets == 0)).sum())  # 统计 TN。
    precision = true_positive / max(1, true_positive + false_positive)  # 计算 Precision 并避免除零。
    recall = true_positive / max(1, true_positive + false_negative)  # 计算 Recall。
    accuracy = (true_positive + true_negative) / len(targets)  # 计算 Accuracy。
    return {"tp": true_positive, "fp": false_positive, "fn": false_negative, "tn": true_negative, "precision": precision, "recall": recall, "accuracy": accuracy}  # 返回完整指标。

def train_model(loss_kind, steps=300, learning_rate=0.2):  # 用相同初始化训练普通 BCE 或 Focal 模型。
    weight = torch.nn.Parameter(torch.tensor([[0.0]]))  # 初始化一维权重。
    bias = torch.nn.Parameter(torch.tensor([-1.8]))  # 用低先验初始化偏置模拟少数类。
    optimizer = torch.optim.SGD([weight, bias], lr=learning_rate)  # 使用基础 SGD 更新参数。
    history = []  # 保存训练损失轨迹。
    for step in range(steps):  # 执行固定步数全批训练。
        logits = (features @ weight).squeeze(1) + bias  # 计算每笔交易 Logit。
        probabilities = torch.sigmoid(logits)  # 转为欺诈概率。
        if loss_kind == "bce":  # 普通 BCE 对所有样本平均。
            losses = -(labels * torch.log(probabilities + 1e-8) + (1 - labels) * torch.log(1 - probabilities + 1e-8))  # 手写逐样本 BCE。
        else:  # 手写带 Alpha 的 Focal Loss。
            target_probability = labels * probabilities + (1 - labels) * (1 - probabilities)  # 计算正确类别概率 p_t。
            alpha_weight = labels * 0.80 + (1 - labels) * 0.20  # 给少数正类更高 Alpha。
            losses = -alpha_weight * (1 - target_probability).pow(2.0) * torch.log(target_probability + 1e-8)  # 应用 gamma=2 聚焦难例。
        loss = losses.mean()  # 对全部交易取平均损失。
        optimizer.zero_grad()  # 清空上一轮梯度。
        loss.backward()  # 反向传播到权重和偏置。
        optimizer.step()  # 更新 Logistic 参数。
        if step in {0, 49, 149, 299}:  # 只保存四个代表性训练点。
            history.append((step + 1, float(loss.detach()), float(weight.detach()), float(bias.detach())))  # 记录损失和参数。
    final_probabilities = torch.sigmoid((features @ weight).squeeze(1) + bias).detach()  # 计算训练后概率。
    return final_probabilities, history  # 返回概率和学习轨迹。

bce_probabilities, bce_history = train_model("bce")  # 训练普通 BCE 模型。
print(f"全负类 Accuracy={all_negative_accuracy:.1%}，Recall={all_negative_recall:.1%}")  # 展示 Accuracy 陷阱。
print("BCE 训练轨迹：", bce_history)  # 展示普通损失收敛和参数变化。

全负类 Accuracy=85.7%，Recall=0.0%
BCE 训练轨迹： [(1, 0.4101204574108124, 0.007914691232144833, -1.7997987270355225), (50, 0.39725810289382935, 0.3547813296318054, -1.8612195253372192), (150, 0.3762008249759674, 0.9523146748542786, -2.1114578247070312), (300, 0.35137271881103516, 1.7290860414505005, -2.4845073223114014)]


### 核心实现：观察 Focal 的逐样本降权机制

In [3]:
focal_probabilities, focal_history = train_model("focal")  # 使用同一数据训练 Focal 模型。
with torch.no_grad():  # 关闭梯度以分析最终样本权重。
    target_probability = labels * focal_probabilities + (1 - labels) * (1 - focal_probabilities)  # 计算每个样本正确类概率。
    modulation = (1 - target_probability).pow(2.0)  # 计算 gamma=2 的难例调制因子。
analysis_indices = [0, 10, 29, 30, 31, 34]  # 选择易负类、难负类和五个正类中的代表样本。
print("index  risk  label  probability  p_t    focal_weight")  # 输出逐样本机制表头。
for index in analysis_indices:  # 逐代表样本展示权重。
    print(f"{index:>3}   {features[index, 0].item():.2f}    {int(labels[index])}      {focal_probabilities[index].item():.3f}    {target_probability[index].item():.3f}     {modulation[index].item():.3f}")  # 展示易例权重接近零、难例权重大。
print("Focal 训练轨迹：", focal_history)  # 展示手写 Focal Loss 的真实优化过程。

index  risk  label  probability  p_t    focal_weight
  0   0.05    0      0.336    0.664     0.113
 10   0.30    0      0.403    0.597     0.163
 29   0.72    0      0.523    0.477     0.274
 30   0.55    1      0.474    0.474     0.277
 31   0.61    1      0.491    0.491     0.259
 34   0.90    1      0.574    0.574     0.181
Focal 训练轨迹： [(1, 0.16489467024803162, 0.016580572351813316, -1.7765079736709595), (50, 0.053822021931409836, 0.6037677526473999, -0.9653086066246033), (150, 0.03804062679409981, 0.9522567391395569, -0.6648395657539368), (300, 0.03647931292653084, 1.152689814567566, -0.7375658750534058)]


## 结果解读：Loss 与业务阈值分别选择

In [4]:
def threshold_search(probabilities, targets, false_negative_cost=8, false_positive_cost=1):  # 按业务成本搜索验证阈值。
    rows = []  # 收集每个候选阈值的指标和成本。
    for threshold_integer in range(10, 91, 5):  # 从 0.10 到 0.90 扫描阈值。
        threshold = threshold_integer / 100  # 转换为概率阈值。
        metrics = binary_metrics(probabilities, targets, threshold)  # 计算混淆矩阵。
        cost = metrics["fn"] * false_negative_cost + metrics["fp"] * false_positive_cost  # 按漏报八倍代价计算成本。
        rows.append({"threshold": threshold, "cost": cost, **metrics})  # 保存全部结果。
    return min(rows, key=lambda row: (row["cost"], -row["recall"], row["threshold"])), rows  # 选择最低成本且高召回的阈值。

bce_half = binary_metrics(bce_probabilities, labels, 0.5)  # 评估普通 BCE 固定 0.5。
focal_half = binary_metrics(focal_probabilities, labels, 0.5)  # 评估 Focal 固定 0.5。
best_threshold, threshold_rows = threshold_search(focal_probabilities, labels)  # 在 Focal 概率上搜索业务阈值。
print("方案                  threshold  TP FP FN TN  precision recall accuracy cost")  # 输出同口径结果表头。
for name, threshold, metrics in [("BCE固定", 0.5, bce_half), ("Focal固定", 0.5, focal_half), ("Focal成本阈值", best_threshold["threshold"], best_threshold)]:  # 展示三种训练/决策组合。
    cost = metrics["fn"] * 8 + metrics["fp"]  # 计算统一业务成本。
    print(f"{name:<20} {threshold:>8.2f}  {metrics['tp']:>2} {metrics['fp']:>2} {metrics['fn']:>2} {metrics['tn']:>2}  {metrics['precision']:>9.2f} {metrics['recall']:>6.2f} {metrics['accuracy']:>8.2f} {cost:>4}")  # 输出真实对照。
print("阈值附近候选：", [row for row in threshold_rows if abs(row["threshold"] - best_threshold["threshold"]) <= 0.05])  # 展示最优点附近权衡。
print("解读：Focal 改变训练中样本贡献，阈值再把概率转换为业务动作；两者不能混为一谈。")  # 解释双层决策。

方案                  threshold  TP FP FN TN  precision recall accuracy cost
BCE固定                    0.50   0  0  5 30       0.00   0.00     0.86   40
Focal固定                  0.50   3  3  2 27       0.50   0.60     0.86   19
Focal成本阈值                0.45   5 11  0 19       0.31   1.00     0.69   11
阈值附近候选： [{'threshold': 0.4, 'cost': 20, 'tp': 5, 'fp': 20, 'fn': 0, 'tn': 10, 'precision': 0.2, 'recall': 1.0, 'accuracy': 0.42857142857142855}, {'threshold': 0.45, 'cost': 11, 'tp': 5, 'fp': 11, 'fn': 0, 'tn': 19, 'precision': 0.3125, 'recall': 1.0, 'accuracy': 0.6857142857142857}, {'threshold': 0.5, 'cost': 19, 'tp': 3, 'fp': 3, 'fn': 2, 'tn': 27, 'precision': 0.5, 'recall': 0.6, 'accuracy': 0.8571428571428571}]
解读：Focal 改变训练中样本贡献，阈值再把概率转换为业务动作；两者不能混为一谈。


## 失败案例：Gamma 过大让中等难度样本几乎失去梯度

In [5]:
with torch.no_grad():  # 在固定最终概率上比较不同 Gamma 的调制权重。
    gamma_two = (1 - target_probability).pow(2.0)  # 计算常用 gamma=2 权重。
    gamma_eight = (1 - target_probability).pow(8.0)  # 构造过度聚焦 gamma=8。
medium_index = 31  # 选择风险 0.61 的边界正例。
easy_negative_index = 0  # 选择风险最低的易负例。
print(f"边界正例 index={medium_index} p_t={target_probability[medium_index]:.3f} gamma2={gamma_two[medium_index]:.6f} gamma8={gamma_eight[medium_index]:.6f}")  # 展示权重大幅缩小。
print(f"易负例 index={easy_negative_index} p_t={target_probability[easy_negative_index]:.3f} gamma2={gamma_two[easy_negative_index]:.6f} gamma8={gamma_eight[easy_negative_index]:.6f}")  # 展示易例几乎完全消失。
gamma_ratio = gamma_eight[medium_index].item() / gamma_two[medium_index].item()  # 计算边界正例梯度调制比例。
print(f"边界正例 gamma8/gamma2={gamma_ratio:.3%}")  # 量化过大 Gamma 的风险。
print("生产边界：Alpha/Gamma/阈值只在验证集调；上线监控 PR、Recall、校准、群体差异和先验漂移，不能在测试集反复选参。")  # 总结工程边界。

边界正例 index=31 p_t=0.491 gamma2=0.258680 gamma8=0.004478
易负例 index=0 p_t=0.664 gamma2=0.113082 gamma8=0.000164
边界正例 gamma8/gamma2=1.731%
生产边界：Alpha/Gamma/阈值只在验证集调；上线监控 PR、Recall、校准、群体差异和先验漂移，不能在测试集反复选参。


## 回归测试：最后只保护不平衡陷阱、梯度聚焦与成本阈值

In [6]:
assert all_negative_accuracy > 0.80 and all_negative_recall == 0.0  # 验证高 Accuracy 可与零少数类召回同时存在。
assert gamma_two[easy_negative_index] < gamma_two[medium_index]  # 验证 Focal 对易负例降权更多。
assert best_threshold["recall"] >= focal_half["recall"]  # 验证高漏报成本阈值不降低少数类召回。
assert best_threshold["cost"] <= focal_half["fn"] * 8 + focal_half["fp"]  # 验证搜索阈值不高于固定 0.5 业务成本。
assert gamma_ratio < 0.10  # 验证 gamma=8 会把本例边界正例权重压到 gamma=2 的十分之一以下。
print("回归测试通过：Accuracy 陷阱、Focal 难例权重、召回、业务成本和 Gamma 过大反例均成立。")  # 用少量断言总结不平衡合同。

回归测试通过：Accuracy 陷阱、Focal 难例权重、召回、业务成本和 Gamma 过大反例均成立。
